# 读取LigandMPNN的结果csv文件，按照设计的个数进行排序，取前十的序列构建Protenix的输入json文件

In [17]:
# 读取csv文件，将其中sequence列的序列提取出来，保存在一个list中
import os
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import argparse
import pandas as pd
import json
import os


json_temple_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/template_without_msa.json"
pdb_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Merged_PDBs"

pdb = '5d94'
res = '030'
temperature = '0.2'
path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/{pdb}-{temperature}T/ligandmpnn_v_32_{res}_25'



#根据template.json的内容以及fasta文件，编写json文件，将fasta中的pro序列替换template.json中的pro_sequence，将fasta中的pep序列替换template.json中的pep_sequence，保存为新的json文件

with open(json_temple_path, 'r') as file:
    tmpl = json.load(file)

jobs = []
df = pd.read_csv(f'{path}/5d94_rescored_top10.csv')


for _, row in df.iloc[:10].iterrows():
    name = row.iloc[0]
    seq_pro = row.iloc[1]
    seq_pep = row.iloc[2]

    job = json.loads(json.dumps(tmpl[0]))
    job['name'] = pdb + "_" + str(name)
    job['sequences'][0]['proteinChain']['sequence'] = seq_pro
    job['sequences'][1]['proteinChain']['sequence'] = seq_pep
    jobs.append(job.copy())


with open(f'{path}/protenix_pred-filter2.json', 'w') as f:
    f.write(json.dumps(jobs, indent=4))

In [4]:
# 读取csv文件，将其中sequence列的序列提取出来，保存在一个list中
import os
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import argparse
import pandas as pd
import json
import os


json_temple_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/template_without_msa.json"
pdb_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Merged_PDBs"

pdb = '5d94'
res = '030'
temperature = '0.2'
path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/{pdb}-{temperature}T/ligandmpnn_v_32_{res}_25'



#根据template.json的内容以及fasta文件，编写json文件，将fasta中的pro序列替换template.json中的pro_sequence，将fasta中的pep序列替换template.json中的pep_sequence，保存为新的json文件

with open(json_temple_path, 'r') as file:
    tmpl = json.load(file)

jobs = []
df = pd.read_csv(f'{path}/5d94_ranked_by_max_overall_confidence.csv')


for _, row in df.iloc[:10].iterrows():
    name = row.iloc[0]
    seq_pro = row.iloc[1]
    seq_pep = row.iloc[2]

    job = json.loads(json.dumps(tmpl[0]))
    job['name'] = pdb + "_" + str(name)
    job['sequences'][0]['proteinChain']['sequence'] = seq_pro
    job['sequences'][1]['proteinChain']['sequence'] = seq_pep
    jobs.append(job.copy())


with open(f'{path}/protenix_pred-filter3.json', 'w') as f:
    f.write(json.dumps(jobs, indent=4))





# 对Protenix的预测指标进行分析

In [50]:
import os
import pandas as pd
from pathlib import Path
import pandas as pd
from Bio.PDB import PDBParser, FastMMCIFParser, Superimposer

filters = [1,2,3]
reses = ['020', '030']
temperatures = ['0.1','0.2']

merged_pdb_path = Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Merged_PDBs/5d94.pdb")
pdbparser = PDBParser(QUIET=True)
mmcifparser = FastMMCIFParser(QUIET=True)
structure_ref = pdbparser.get_structure('ref', merged_pdb_path)
chain_L_ref = structure_ref[0]['L']
ref_atoms = [atom for atom in chain_L_ref.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]


for filter in filters:
    for res in reses:
        for temperature in temperatures:
            base_dir = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-{temperature}T/ligandmpnn_v_32_{res}_25/predict_output-filter{filter}"
            rows = []

            for root, dirs, files in os.walk(base_dir):
                seed = root.split("/")[-2]
                sample_name = root.split("/")[-3]

                candidates = []
                if not root.endswith("predictions"):
                    continue
                
                metrics_by_id = {}

                for file in files:
                    if file.endswith(".json"):
                        sample_id = file.split("_")[-1].split(".")[0]
                        json_path = os.path.join(root, file)
                        if not os.path.exists(json_path):
                            continue
                        df = pd.read_json(json_path)
                        metrics_by_id.setdefault(sample_id, {}).update(
                            {
                                "plddt": round(float(df["plddt"].iloc[0]), 4),
                                "gpde": round(float(df["gpde"].iloc[0]), 4),
                                "ptm": round(float(df["ptm"].iloc[0]), 4),
                                "iptm": round(float(df["iptm"].iloc[0]), 4),
                                "ranking_score": round(float(df["ranking_score"].iloc[0]), 4),
                            }
                        )

                    elif file.endswith(".cif"):
                        sample_id = file.split("_")[-1].split(".")[0]
                        cif_path = os.path.join(root, file)
                        if not os.path.exists(cif_path):
                            continue
                        structure_pred = mmcifparser.get_structure('pred', cif_path)
                        chain_L_pred = structure_pred[0]['B']
                        pred_atoms = [atom for atom in chain_L_pred.get_atoms() if atom.get_id() in ['N', 'CA', 'C', 'O']]

                        sup = Superimposer()
                        sup.set_atoms(ref_atoms, pred_atoms)
                        scRMSD = round(sup.rms, 4)
                        metrics_by_id.setdefault(sample_id, {})["scRMSD"] = scRMSD

                for sample_id, data in metrics_by_id.items():
                    required_keys = {"plddt", "gpde", "ptm", "iptm", "ranking_score", "scRMSD"}
                    if not required_keys.issubset(data):
                        continue
                    candidates.append(
                        (
                            sample_name,
                            seed,
                            sample_id,
                            data["plddt"],
                            data["gpde"],
                            data["ptm"],
                            data["iptm"],
                            data["ranking_score"],
                            data["scRMSD"],
                        )
                    )

                # flatten all candidate rows across directories
                rows.extend(candidates)

            result = pd.DataFrame(rows, columns=["complex", "seed", "id", "plddt", "gpde", "ptm", "iptm", "ranking_score", "scRMSD"])
            result.to_csv(f"{base_dir}/metrics_summary-filter{filter}.csv", index=False)
            result

            # # 对result按照相同sample下不同seed的ranking_score取最大值，将对应的行保存到新的df中
            # if not result.empty:
            #     result = result.sort_values("ranking_score", ascending=False).reset_index(drop=True)
            #     unique_rows = []
            #     seen_samples = set()
            #     for _, row in result.iterrows():
            #         complex_name = row["complex"]
            #         if complex_name not in seen_samples:
            #             unique_rows.append(row)
            #             seen_samples.add(complex_name)
            #     result_unique = pd.DataFrame(unique_rows)
            #     result_unique = result_unique.reset_index(drop=True)
            # result_unique.to_csv(f"{base_dir}/protenix_summary_unique-filter{filter}.csv", index=False)
            # result_unique

            # 过滤scRMSD大于1的结果，保存到新的df中
            result_filtered = result[result['scRMSD'] <= 1.0].reset_index(drop=True)
            result_filtered.to_csv(f"{base_dir}/metrics_filtered_scRMSD_le1-filter{filter}.csv", index=False)
            result_filtered

In [ ]:
pdbs = ['5d94']
for pdb in pdbs:
    fil_df = pd.read_csv(f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/{pdb}-0.1T/ligandmpnn_v_32_020_25/predict_output-filter1/metrics_filtered_scRMSD_le1-filter1.csv')
    df = pd.read_csv(f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/{pdb}-0.1T/ligandmpnn_v_32_020_25/predict_output-filter1/metrics_summary-filter1.csv')
    # 筛选fil_df中plddt大于85，iptm大于0.7的行中complex不重复的个数，将其保存在一个字典中，key为pdb，value为个数
    count = fil_df[(fil_df['plddt'] > 85) & (fil_df['iptm'] > 0.7)]['complex'].nunique()
    oricount = df['complex'].nunique()
    print(f"{pdb}: {count}, decoys before filtering: {oricount}, success rate: {count/oricount:.2%}")

,complex,seed,id,plddt,gpde,ptm,iptm,ranking_score,scRMSD
0,5d94_6,seed_44,2,88.1597,0.5350,0.9115,0.7621,0.7920,0.8512
1,5d94_17,seed_44,2,86.4392,0.6085,0.9006,0.7043,0.7435,0.5350
2,5d94_17,seed_43,0,87.0814,0.5910,0.9056,0.7204,0.7575,0.9931
3,5d94_18,seed_44,1,86.9587,0.5512,0.9066,0.7656,0.7938,0.4523
4,5d94_18,seed_43,3,88.2540,0.5097,0.9137,0.7865,0.8120,0.8082
5,5d94_10,seed_44,3,84.5651,0.6573,0.8798,0.4198,0.5118,0.7639
6,5d94_10,seed_44,4,84.3888,0.6565,0.8815,0.4122,0.5061,0.7995
7,5d94_10,seed_43,1,84.2279,0.6752,0.8857,0.5266,0.5984,0.7523
8,5d94_10,seed_43,2,83.6882,0.6862,0.8827,0.5209,0.5933,0.6508
9,5d94_10,seed_43,3,83.6596,0.6892,0.8832,0.5170,0.5902,0.5076
